<a href="https://colab.research.google.com/github/davidlealo/sic_ia_2026/blob/main/clases/sic_2026_ia_Clase_21.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Revisión de grupos



# Datos sintéticos para clasificar cantos de aves chilenas

Material de apoyo para estudiantes. Resume una conversación de trabajo sobre el modelo EfficientNet-B1-BirdSet-XCL, la generación de datos sintéticos de cantos de aves y el diseño de un experimento para evaluar si el audio sintético mejora la clasificación de especies chilenas con pocas grabaciones.

Fecha: noviembre de 2026.

---

## 1. Qué es EfficientNet-B1-BirdSet-XCL

Es un clasificador de cantos de aves publicado por el DBD-research-group como parte del benchmark **BirdSet** (Rauch et al., ICLR 2025).

- **Arquitectura:** adapta EfficientNet-B1 para clasificar segmentos de audio de 5 segundos en 9.736 especies y produce embeddings de 1.280 dimensiones. En Hugging Face aparece etiquetado como "Image Classification" porque opera sobre espectrogramas, que trata como imágenes.
- **Datos de entrenamiento (XCL):** Xeno-Canto Large, con alrededor de 530 mil grabaciones de unas 10.000 especies. Los pesos se inicializan desde ImageNet y luego se entrenan de forma supervisada sobre XCL. Existe también una variante **XCM**, entrenada con una versión mediana del dataset.
- **Contexto del benchmark:** BirdSet entrena con grabaciones focales (una especie dominante por grabación) y evalúa en paisajes sonoros reales, donde la tarea es multi-etiqueta. El salto entre ambos dominios es lo que el benchmark busca medir.
- **Familia de modelos:** el mismo grupo publica versiones XCL y XCM de Wav2Vec2-Base, EfficientNet-B1, AST y ConvNeXT-Base. EfficientNet-B1 es la opción más liviana (del orden de 7,8 millones de parámetros en la arquitectura base), útil para inferencia en hardware modesto o como extractor de embeddings.
- **Advertencia práctica:** el modelo espera audio preprocesado según BirdSet (32 kHz y espectrograma mel con parámetros específicos). Conviene usar la librería `birdset` o replicar exactamente ese preprocesamiento; de lo contrario, el rendimiento cae bastante.

---

## 2. Puede este modelo generar sonidos

No. Es un modelo discriminativo: recibe un espectrograma y devuelve probabilidades por especie. No tiene decodificador que produzca audio. Sin embargo, cumple roles útiles en un pipeline de datos sintéticos:

1. **Filtro de calidad:** se clasifica cada muestra sintética y se descartan las que no se reconocen como la especie objetivo o tienen baja confianza. Solo funciona si la especie está entre sus 9.736 clases; algunas aves chilenas raras o endémicas pueden faltar o tener pocas grabaciones en Xeno-Canto.
2. **Métrica de fidelidad:** sus embeddings permiten comparar la distribución de audio real con la de audio sintético, al estilo de la Fréchet Audio Distance, pero en un espacio de representación específico de aves.
3. **Backbone del clasificador final:** lo más probable es que el experimento termine afinando un modelo como este, no entrenando uno desde cero.

---

## 3. Estrategias para aumentar datos

De menor a mayor complejidad:

1. **Aumentación clásica:** cambio de tono leve, estiramiento temporal, ganancia, SpecAugment y, sobre todo, mezcla de vocalizaciones con ruido de fondo real de hábitats chilenos (viento, lluvia, otras especies, insectos). Suele ser lo que más rinde, porque el problema central es el salto entre grabaciones focales limpias y paisajes sonoros ruidosos.
2. **Recombinación de sílabas:** segmentar cantos en unidades y armar secuencias nuevas. Funciona en especies con repertorio estructurado, aunque puede producir secuencias biológicamente inverosímiles.
3. **Modelos generativos:** GANs sobre espectrogramas o modelos de difusión condicionados, idealmente partiendo de un modelo preentrenado y afinándolo con las grabaciones disponibles.

**Riesgo principal de los modelos generativos:** con muy pocas grabaciones por especie, tienden a memorizar. Producen variaciones cosméticas de las mismas grabaciones y no diversidad real, lo que puede inflar artificialmente los resultados.

---

## 4. Diseño experimental recomendado

- **Separar por grabación, no por segmento.** Si se generan sintéticos a partir de una grabación, esa grabación y todo lo derivado de ella no pueden aparecer en el conjunto de prueba. Es el error más común en este tipo de estudios.
- **Evaluar solo con audio real**, idealmente paisajes sonoros chilenos y no grabaciones focales.
- **Comparar contra baselines fuertes:**
  - solo datos reales;
  - datos reales con aumentación clásica;
  - datos reales con sintéticos;
  - clasificación few-shot con embeddings congelados (prototipos por especie), que con pocos datos suele ser difícil de superar.
- **Barrer la proporción real/sintético** (por ejemplo 1:0, 1:1, 1:5, 1:10) para identificar dónde la ganancia se estabiliza o empieza a degradar.
- **Medir por especie**, no solo el promedio.

**Criterio de interpretación:** si los sintéticos superan a "solo reales" pero no a "aumentación clásica", la conclusión honesta es que el aporte proviene de la regularización y no de la síntesis en sí.

---

## 5. Modelos generativos abiertos recomendados

### 5.1 ECOGEN (recomendado para empezar)

Desarrollado en la Université de Moncton y publicado en *Methods in Ecology and Evolution*. Su objetivo explícito es aumentar las muestras de clases de cantos subrepresentadas. Genera espectrogramas con técnicas de generación de imágenes y luego los convierte de vuelta a audio. Los autores reportan que añadir muestras de ECOGEN mejoró la precisión de un clasificador en 12% en promedio y superó a la aumentación clásica en el 80% de los casos. Es open source y funciona en hardware básico.

Observación: ese 80% implica que en uno de cada cinco casos la aumentación clásica fue igual o mejor, por lo que el baseline clásico sigue siendo imprescindible.

### 5.2 BirdDiff (más controlable, más exigente)

Basado en DiffWave, un modelo de difusión que genera directamente la forma de onda. Se condiciona con coeficientes MFCC, etiquetas de especie y descripciones textuales, y está pensado para grabaciones de campo ruidosas. El código y el dataset están disponibles en GitHub.

Advertencia de los propios autores: sin una etapa previa de mejora de la relación señal/ruido, el modelo no genera cantos inteligibles ni específicos de la especie.

### 5.3 Difusión sobre espectrogramas con Stable Diffusion

Un trabajo de 2024 mostró que aumentar la clasificación de especies con espectrogramas sintéticos generados con Stable Diffusion mejora el rendimiento en ambientes ruidosos frente a usar solo los datos originales. Es una buena opción si ya se maneja ese ecosistema y se quiere afinar con LoRA.

### 5.4 Alternativa general

Stable Audio Open es un modelo de texto a audio con pesos públicos que puede afinarse, pero no está entrenado específicamente en aves y su licencia comunitaria tiene condiciones de uso que deben revisarse antes de publicar.

---

## 6. Ruta sugerida

1. Baseline con aumentación clásica y mezcla con ruido de fondo chileno.
2. ECOGEN como primer generador, por bajo costo y evidencia previa.
3. BirdDiff como generador fuerte, si los resultados de ECOGEN lo justifican.
4. Evaluación con EfficientNet-B1-BirdSet-XCL (o ConvNeXt de BirdSet) afinado, probando siempre solo con grabaciones reales separadas por grabación.

Con muy pocas grabaciones por especie (por ejemplo, menos de 10 a 20), conviene preentrenar el generador con muchas especies, por ejemplo aves sudamericanas de Xeno-Canto, y luego condicionar o afinar para las especies chilenas escasas, en lugar de entrenar un generador por especie.

---

## Referencias y enlaces

### Modelo y benchmark

- Modelo en Hugging Face: [DBD-research-group/EfficientNet-B1-BirdSet-XCL](https://huggingface.co/DBD-research-group/EfficientNet-B1-BirdSet-XCL)
- Colección de modelos BirdSet: [DBD-research-group en Hugging Face](https://huggingface.co/DBD-research-group)
- Rauch, L. et al. BirdSet: A Large-Scale Dataset for Audio Classification in Avian Bioacoustics. ICLR 2025. [arXiv:2403.10380](https://arxiv.org/abs/2403.10380)
- Presentación de BirdSet en ICLR 2025: [diapositivas (PDF)](https://iclr.cc/media/iclr-2025/Slides/28986.pdf)
- Resumen técnico del modelo: [BirdSetEfficientNetB1 en Emergent Mind](https://www.emergentmind.com/topics/birdsetefficientnetb1)
- Uncertainty Calibration of Multi-Label Bird Sound Classifiers (compara modelos BirdSet y Perch v2): [arXiv:2511.08261](https://arxiv.org/abs/2511.08261)

### Generación de cantos de aves

- Guei, A.-C., Christin, S., Lecomte, N. y Hervet, É. (2024). ECOGEN: Bird sounds generation using deep learning. *Methods in Ecology and Evolution*, 15(1), 69-79. [doi:10.1111/2041-210X.14239](https://doi.org/10.1111/2041-210X.14239)
- Nota de prensa sobre ECOGEN: [NSERC](https://nserc-crsng.canada.ca/en/new-deep-learning-ai-tool-helps-ecologists-monitor-rare-birds-through-their-songs)
- Song, T. et al. Towards High-Fidelity and Controllable Bioacoustic Generation via Enhanced Diffusion Learning (BirdDiff). [arXiv:2509.00318](https://arxiv.org/abs/2509.00318) y [versión en ScienceDirect](https://www.sciencedirect.com/science/article/pii/S1574954126003249)
- Código y datos de BirdDiff: [GitHub](https://github.com/BirdDiff/Bioacoustics-Generation/releases/tag/V1)
- Generative AI-based data augmentation for improved bioacoustic classification in noisy environments: [arXiv:2412.01530](https://arxiv.org/abs/2412.01530)

### Otros recursos

- Xeno-Canto, repositorio colaborativo de grabaciones de aves: [xeno-canto.org](https://xeno-canto.org)
- Kong, Z. et al. DiffWave: A Versatile Diffusion Model for Audio Synthesis. ICLR 2021. [arXiv:2009.09761](https://arxiv.org/abs/2009.09761)